# Multi-Tenant Governance for LangChain

This cookbook shows one **shared** LangChain/LangGraph agent serving three tenants — Healthcare, Fintech, and Enterprise — each governed by a different policy, selected at invocation time from `runtime.context.tenant_id`. It demonstrates:

- a single compiled LangGraph agent, invoked repeatedly for different tenants, never recreated;
- tenant policy resolution driven by `tenant_id` carried on LangGraph's real `Runtime.context`;
- `langchain-tealtiger`'s `TealTigerMiddleware` for deterministic governance: input-defense PII blocking (`before_model`), tool-call authorization (`wrap_tool_call`: allowlist, FREEZE, transaction cap), and `MONITOR` mode;
- TealTiger's public `BudgetManager`/`CostTracker` cost APIs for a per-tenant cost cap;
- tenant- and session-isolated audit evidence;
- policy switching across tenants without rebuilding the shared agent object.

Running this example currently requires two small fixes to `packages/langchain-tealtiger`, included alongside it and covered by regression tests (see `tests/test_engine.py`, `tests/test_middleware.py`). They are summarized in section 2 and in *Production notes* at the end. No monkey-patching or private-module access is used anywhere in this notebook — every governance decision below is produced by the public `langchain_tealtiger.TealTigerMiddleware` API, backed by `tealtiger.guardrails.PIIDetectionGuardrail` and `tealtiger.BudgetManager` from the published `tealtiger` package.

## 1. Install dependencies

Same version pins as the existing `budget_research_agent.ipynb` cookbook, plus this package's own local source (includes the fixes described in section 2).

In [ ]:
%pip install -q "tealtiger==1.4.0" "langgraph==1.2.11" "langchain-core>=0.3.0"
%pip install -q -e ../../packages/langchain-tealtiger

## 2. Required package fixes

`TealTigerMiddleware` could not previously be constructed, and its documented `pii` policy type was never implemented. Both are fixed in the package itself, not worked around in this notebook:

1. **Constructor mismatch.** `middleware.py` forwarded an `otel_enabled` keyword to `GovernanceBridge(...)`, which did not accept it, so construction always raised `TypeError`. `GovernanceBridge.__init__` now accepts and stores `otel_enabled`.
2. **Missing circuit-breaker bookkeeping.** `wrap_tool_call`'s success/failure paths called `record_tool_success`/`record_tool_failure`, which were never defined. Both now implement the documented `circuit_breaker` policy (`failure_threshold`): consecutive failures are tracked per tool, and the breaker opens (denies) once the threshold is reached, closing again on the next success. See `TestGovernanceBridgeCircuitBreaker` (`test_engine.py`) and `TestCircuitBreakerViaMiddleware` (`test_middleware.py`).
3. **Missing PII evaluation.** `before_model`/`after_model`/`after_tool` called `evaluate_content`, which did not exist, and referenced `GovernanceAction.REDACT`, which was not a member of the `GovernanceAction` enum. `GovernanceAction.REDACT` and `GovernanceDecision.redacted_content` were added, and `evaluate_content` is implemented against `tealtiger.guardrails.PIIDetectionGuardrail` from the published `tealtiger` package. See `TestGovernanceBridgeEvaluateContent`, `TestGovernanceBridgeSummaryRedact` (`test_engine.py`), and `TestPIIInputDefense` (`test_middleware.py`).

`wrap_tool_call`'s allowlist/FREEZE/rate_limit/cost_limit evaluation order and semantics are unchanged. `packages/langchain-tealtiger/tests/` passes in full: 65 tests (39 pre-existing, 26 new).

The `mode` property setter (`set_mode`), `after_agent`'s `finalize_session`, and `export_evidence` still call undefined `GovernanceBridge` methods and are not addressed here; this notebook does not call any of them — tenant/session routing uses multiple pre-built middleware instances (section 6) instead of mutating one instance's mode, and evidence is read via the `.evidence`/`.summary` properties.

## 3. Imports

In [ ]:
from __future__ import annotations

import uuid
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from types import SimpleNamespace
from typing import Literal, TypedDict

from langchain_core.messages import HumanMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime

from langchain_tealtiger import GovernanceAction, GovernanceMode, TealTigerMiddleware
from tealtiger import BudgetManager, BudgetScope, CostBreakdown, CostRecord, TokenUsage
from tealtiger.cost.storage import InMemoryCostStorage

print("langchain_tealtiger and tealtiger imported; TealTigerMiddleware constructs via its normal public API.")

## 4. Tenant context schema

`context_schema` and `Runtime.context` are real, general-purpose LangGraph 1.x features (not TealTiger-specific) — the same `Runtime` type `langchain_tealtiger.middleware` itself imports and type-hints on every lifecycle hook. `tenant_id` is the actual policy selector: it is the only field `resolve_governance` (section 6) reads to decide which policy configuration applies. `session_id` is a notebook-defined identifier used to scope audit-evidence isolation between concurrent sessions of the same tenant; TealTiger has no `session_id` concept of its own — only a per-decision `correlation_id`, which is real, generated by `GovernanceBridge.evaluate()`/`evaluate_content()`, and used below.

In [ ]:
@dataclass
class TenantContext:
    tenant_id: str
    session_id: str


HEALTHCARE_TENANT = "healthcare-001"
FINTECH_TENANT = "fintech-001"
ENTERPRISE_TENANT = "enterprise-001"

## 5. Tenant policy definitions

Each tenant maps to a plain configuration dict, authored using the policy types `GovernanceBridge._parse_policies` understands: `tool_allowlist`, `rate_limit`, `pii`, and `freeze_tools`/`mode` (constructor-level). There is no HIPAA, fintech, or enterprise policy pack anywhere in this repository (`policy-packs/` only has MCP-tool-scoped packs) — these are authored by hand to model each tenant's intent using supported policy types.

- **Healthcare** — `pii: {"action": "block"}` (PII blocking, section 2) plus a strict allowlist and `freeze_tools` for every other tool (an immutable deny, independent of mode).
- **Fintech** — `rate_limit.max_calls` for a transaction cap, plus a separate `BudgetManager` budget for a cost cap. `rate_limit.max_calls` is a **session-lifetime call-count cap**, not a time-windowed rate limiter — `window` is accepted by the parser but never read again anywhere in the engine.
- **Enterprise** — broad allowlist, `GovernanceMode.MONITOR` so violations are recorded but never block execution.

In [ ]:
TENANT_POLICIES: dict[str, dict] = {
    HEALTHCARE_TENANT: {
        "label": "Healthcare (PII blocking + strict access control / HIPAA-oriented audit)",
        "policies": [
            {"type": "tool_allowlist", "tools": ["lookup_patient_record"]},
            {"type": "pii", "action": "block"},
        ],
        "freeze_tools": ["process_transaction", "generate_report"],
        "mode": GovernanceMode.ENFORCE,
        "budget_limit_usd": None,
    },
    FINTECH_TENANT: {
        "label": "Fintech (cost cap + transaction limit)",
        "policies": [
            {"type": "tool_allowlist", "tools": ["process_transaction"]},
            {"type": "rate_limit", "max_calls": 3, "window": "1h"},  # window is parsed but not enforced
        ],
        "freeze_tools": [],
        "mode": GovernanceMode.ENFORCE,
        "budget_limit_usd": 1.00,
    },
    ENTERPRISE_TENANT: {
        "label": "Enterprise (full audit trail + MONITOR mode)",
        "policies": [
            {"type": "tool_allowlist", "tools": ["generate_report"]},
        ],
        "freeze_tools": [],
        "mode": GovernanceMode.MONITOR,
        "budget_limit_usd": None,
    },
}

for _tenant_id, _config in TENANT_POLICIES.items():
    print(f"{_tenant_id}: {_config['label']} (mode={_config['mode'].value})")

## 6. Governance routing / orchestration

`TealTigerMiddleware` has no built-in tenant- or session-aware policy resolution: `policies` are parsed once in `__init__` and there is no supported way to swap them afterward (the `mode` setter calls the undefined `set_mode` and is not used here). All tenant/session routing below is notebook-level orchestration, kept separate from TealTiger's API surface: `tenant_id` selects **which** policy configuration to use; `session_id` scopes a dedicated `TealTigerMiddleware`/`BudgetManager` pair so evidence and counters never leak between concurrent sessions, even for the same tenant. "Policy switching" means *selecting a different, already-constructed governance instance* for a given `(tenant_id, session_id)` — never mutating one instance's policies or mode.

In [ ]:
@dataclass
class TenantGovernanceRuntime:
    tenant_id: str
    session_id: str
    middleware: TealTigerMiddleware
    budget_manager: BudgetManager | None
    storage: InMemoryCostStorage | None
    session_evidence: list[dict] = field(default_factory=list)


_governance_by_session: dict[str, TenantGovernanceRuntime] = {}


def resolve_governance(tenant_id: str, session_id: str) -> TenantGovernanceRuntime:
    """Notebook-level policy resolution: tenant_id is the selector; session_id
    scopes isolation. Not a TealTiger API.
    """
    if session_id in _governance_by_session:
        return _governance_by_session[session_id]

    config = TENANT_POLICIES[tenant_id]
    middleware = TealTigerMiddleware(
        policies=config["policies"],
        agent_id=f"shared-agent::{tenant_id}::{session_id}",
        mode=config["mode"],
        freeze_tools=config["freeze_tools"],
    )
    # before_agent's `runtime` parameter is accepted but unused by the current
    # implementation (only calls self._engine.reset_session()). SimpleNamespace()
    # as a placeholder matches cookbook/langchain/budget_research_agent.ipynb.
    middleware.before_agent({}, SimpleNamespace())

    budget_manager = None
    storage = None
    if config["budget_limit_usd"] is not None:
        storage = InMemoryCostStorage()
        budget_manager = BudgetManager(storage)
        budget_manager.create_budget(
            name=f"{tenant_id}-session-budget",
            limit=config["budget_limit_usd"],
            period="total",
            alert_thresholds=[50, 80, 100],
            action="block",
            # BudgetScope.type is Literal["agent", "project", "organization"] --
            # there is no "tenant" scope, so this uses "agent" with a
            # tenant::session composite id.
            scope=BudgetScope(type="agent", id=f"{tenant_id}::{session_id}"),
        )

    state = TenantGovernanceRuntime(
        tenant_id=tenant_id,
        session_id=session_id,
        middleware=middleware,
        budget_manager=budget_manager,
        storage=storage,
    )
    _governance_by_session[session_id] = state
    return state


def _mirror_new_evidence(
    gov: TenantGovernanceRuntime, tenant_id: str, session_id: str, evidence_count_before: int
) -> None:
    """Copy any newly-recorded decision from the middleware's evidence trail
    into this session's isolated mirror, tagged for display purposes only.
    """
    if len(gov.middleware.evidence) > evidence_count_before:
        latest = gov.middleware.evidence[-1]
        gov.session_evidence.append(
            {**asdict(latest), "tenant_id": tenant_id, "session_id": session_id}
        )

## 7. Shared tools

In [ ]:
def lookup_patient_record(patient_id: str) -> str:
    return f"Patient {patient_id}: last visit 2026-06-02, status: stable."


def process_transaction(account: str, amount_usd: float) -> str:
    return f"Transaction processed for {account}: ${amount_usd:.2f}."


def generate_report(report_type: str) -> str:
    return f"{report_type} report generated with 3 sections."


TOOL_HANDLERS = {
    "lookup_patient_record": lookup_patient_record,
    "process_transaction": process_transaction,
    "generate_report": generate_report,
}

## 8. The shared LangGraph agent

One small graph, compiled exactly once as `shared_agent`. Each node receives the real `Runtime` LangGraph passes it and reads `runtime.context.tenant_id` / `runtime.context.session_id` to resolve the right governance instance. Three stages, matching `TealTigerMiddleware`'s own hook names: `input_defense` calls the real `middleware.before_model` (PII scanning), `preflight_cost` checks the tenant's `BudgetManager` budget, and `call_tool` runs `middleware.wrap_tool_call` (allowlist/FREEZE/transaction-cap enforcement). Every tenant and every demonstration below invokes this same compiled object — it is never rebuilt.

In [ ]:
class GovernanceState(TypedDict):
    messages: list
    tool_name: str
    tool_args: dict
    estimated_cost: float | None
    tool_result: object | None
    denied: bool
    stopped_reason: str | None


def input_defense(state: GovernanceState, runtime: Runtime) -> dict:
    tenant_id = runtime.context.tenant_id
    session_id = runtime.context.session_id
    gov = resolve_governance(tenant_id, session_id)

    count_before = len(gov.middleware.evidence)
    result = gov.middleware.before_model(state, runtime)
    _mirror_new_evidence(gov, tenant_id, session_id, count_before)

    if result and result.get("governance_blocked"):
        return {
            "denied": True,
            "stopped_reason": result.get("reason", "input blocked by governance"),
            "messages": state["messages"],
        }
    if result and result.get("governance_redacted"):
        return {"messages": state["messages"]}
    return {}


async def preflight_cost(state: GovernanceState, runtime: Runtime) -> dict:
    if state.get("denied"):
        return {}

    tenant_id = runtime.context.tenant_id
    session_id = runtime.context.session_id
    gov = resolve_governance(tenant_id, session_id)

    estimated_cost = state.get("estimated_cost")
    if gov.budget_manager is not None and estimated_cost is not None:
        result = await gov.budget_manager.check_budget(
            agent_id=f"{tenant_id}::{session_id}",
            estimated_cost=estimated_cost,
        )
        if not result.allowed:
            return {
                "denied": True,
                "stopped_reason": (
                    f"cost cap exceeded for tenant {tenant_id}: projected "
                    f"${estimated_cost:.2f} would breach the session budget"
                ),
            }
    return {}


async def call_tool(state: GovernanceState, runtime: Runtime) -> dict:
    if state.get("denied"):
        return {}

    tenant_id = runtime.context.tenant_id
    session_id = runtime.context.session_id
    gov = resolve_governance(tenant_id, session_id)

    tool_name = state["tool_name"]
    tool_args = state["tool_args"]
    call_id = str(uuid.uuid4())
    request = SimpleNamespace(tool_call={"id": call_id, "name": tool_name, "args": tool_args})
    handler = TOOL_HANDLERS[tool_name]

    count_before = len(gov.middleware.evidence)
    result = gov.middleware.wrap_tool_call(request, lambda _req: handler(**tool_args))
    _mirror_new_evidence(gov, tenant_id, session_id, count_before)

    denied = (
        isinstance(result, ToolMessage)
        and str(result.content).startswith("[GOVERNANCE DENIED]")
    )

    estimated_cost = state.get("estimated_cost")
    if not denied and gov.budget_manager is not None and estimated_cost is not None:
        record = CostRecord(
            id=str(uuid.uuid4()),
            request_id=call_id,
            agent_id=f"{tenant_id}::{session_id}",
            model="notebook-fixture",
            provider="custom",
            actual_tokens=TokenUsage(input_tokens=0, output_tokens=0, total_tokens=0),
            actual_cost=estimated_cost,
            breakdown=CostBreakdown(input_cost=estimated_cost, output_cost=0.0),
            timestamp=datetime.now(timezone.utc).replace(tzinfo=None).isoformat(),
            metadata={"tool": tool_name},
        )
        await gov.storage.store(record)
        await gov.budget_manager.record_cost(record)

    return {
        "denied": denied,
        "tool_result": str(result.content) if denied else result,
        "stopped_reason": str(result.content) if denied else None,
    }


def route_on_denied(state: GovernanceState) -> Literal["continue", "done"]:
    return "done" if state.get("denied") else "continue"


graph = StateGraph(GovernanceState, context_schema=TenantContext)
graph.add_node("input_defense", input_defense)
graph.add_node("preflight_cost", preflight_cost)
graph.add_node("call_tool", call_tool)
graph.add_edge(START, "input_defense")
graph.add_conditional_edges(
    "input_defense", route_on_denied, {"continue": "preflight_cost", "done": END}
)
graph.add_conditional_edges(
    "preflight_cost", route_on_denied, {"continue": "call_tool", "done": END}
)
graph.add_edge("call_tool", END)

shared_agent = graph.compile()
print("shared_agent compiled once:", shared_agent)

## 9. Helper: invoke the shared agent for a tenant

In [ ]:
async def run_governed_call(
    tenant_id: str,
    session_id: str,
    tool_name: str,
    tool_args: dict,
    estimated_cost: float | None = None,
    user_message: str = "",
) -> dict:
    initial_state: GovernanceState = {
        "messages": [HumanMessage(content=user_message)] if user_message else [],
        "tool_name": tool_name,
        "tool_args": tool_args,
        "estimated_cost": estimated_cost,
        "tool_result": None,
        "denied": False,
        "stopped_reason": None,
    }
    context = TenantContext(tenant_id=tenant_id, session_id=session_id)
    return await shared_agent.ainvoke(initial_state, context=context)

## 10. Healthcare tenant: PII blocking + strict access control

Three calls: a clean request (allowed), a request whose text contains an SSN and email (blocked by the real `pii` policy before any tool is even considered), and an allowed-looking request for a frozen tool (denied unconditionally by FREEZE).

In [ ]:
healthcare_session = "healthcare-session-1"

clean_call = await run_governed_call(
    HEALTHCARE_TENANT, healthcare_session, "lookup_patient_record", {"patient_id": "PT-4821"},
    user_message="Please look up patient PT-4821's latest visit.",
)
print("Clean request:", clean_call["denied"], "->", clean_call["tool_result"])

pii_call = await run_governed_call(
    HEALTHCARE_TENANT, healthcare_session, "lookup_patient_record", {"patient_id": "PT-4821"},
    user_message="Patient SSN is 123-45-6789, please email results to jane.doe@example.com.",
)
print("PII-containing request:", pii_call["denied"], "->", pii_call["stopped_reason"])

frozen_call = await run_governed_call(
    HEALTHCARE_TENANT, healthcare_session, "process_transaction",
    {"account": "acct-1", "amount_usd": 50.0},
    user_message="Please charge $50 to account acct-1.",
)
print("Frozen tool:", frozen_call["denied"], "->", frozen_call["stopped_reason"])

### Healthcare audit evidence

Each decision above is real: produced by `GovernanceBridge.evaluate()`/`evaluate_content()` and recorded in `middleware.evidence`. `<input>` marks a content-stage decision (PII scan); real tool names mark `wrap_tool_call` decisions.

In [ ]:
healthcare_gov = resolve_governance(HEALTHCARE_TENANT, healthcare_session)
for entry in healthcare_gov.session_evidence:
    print(
        f"[{entry['tool_name']}] action={entry['action']} "
        f"reason_codes={entry['reason_codes']} correlation_id={entry['correlation_id']}"
    )

## 11. Fintech tenant: cost cap

Uses the standalone `BudgetManager`/`CostTracker`-family APIs (not `TealTigerMiddleware`'s own `cost_limit` policy, which stays inactive because nothing calls `GovernanceBridge.record_cost` from the middleware hooks). The budget cap is $1.00; each transaction is estimated at $0.60, so the second call is projected to breach the cap and is denied *before* the tool executes.

In [ ]:
fintech_cost_session = "fintech-cost-session"

first = await run_governed_call(
    FINTECH_TENANT, fintech_cost_session, "process_transaction",
    {"account": "acct-9", "amount_usd": 250.0}, estimated_cost=0.60,
)
print("Transaction 1 ($0.60 estimated):", first["denied"], "->", first["tool_result"])

second = await run_governed_call(
    FINTECH_TENANT, fintech_cost_session, "process_transaction",
    {"account": "acct-9", "amount_usd": 250.0}, estimated_cost=0.60,
)
print("Transaction 2 ($0.60 more, breaches $1.00 cap):", second["denied"], "->", second["stopped_reason"])

## 12. Fintech tenant: transaction limit

A **separate session** so this demonstration's call count doesn't interact with the cost-cap demonstration above. `rate_limit.max_calls=3` is a session-lifetime *count* of allowed calls — **not** a sliding- or fixed-time-window rate limiter. The `window: "1h"` value is accepted but never enforced. The first three calls are allowed; the fourth is denied purely because it is the fourth call.

In [ ]:
fintech_tx_session = "fintech-transaction-session"

tx_results = []
for i in range(4):
    res = await run_governed_call(
        FINTECH_TENANT, fintech_tx_session, "process_transaction",
        {"account": "acct-3", "amount_usd": 10.0}, estimated_cost=0.01,
    )
    tx_results.append(res)
    print(f"Call {i + 1}: denied={res['denied']} reason={res['stopped_reason']}")

print("\nThis is a session-level transaction/tool-call cap, not a time-windowed rate limiter.")

## 13. Enterprise tenant: MONITOR mode + full audit trail

Enterprise runs in `GovernanceMode.MONITOR`. In `GovernanceBridge.evaluate()`, a DENY decision is only returned early `if self._mode == GovernanceMode.ENFORCE`; in MONITOR mode every check falls through to a final `ALLOW` decision, but `triggered_policies` still records what *would* have been denied. We call `generate_report` (allowlisted) and then `lookup_patient_record` (not allowlisted for this tenant) to show the violation being recorded while the tool still executes.

In [ ]:
enterprise_session = "enterprise-session-1"

report_result = await run_governed_call(
    ENTERPRISE_TENANT, enterprise_session, "generate_report", {"report_type": "quarterly-compliance"}
)
print("Allowlisted call (generate_report):", report_result["denied"], "->", report_result["tool_result"])

violation_result = await run_governed_call(
    ENTERPRISE_TENANT, enterprise_session, "lookup_patient_record", {"patient_id": "PT-4821"}
)
print(
    "Out-of-policy call under MONITOR (still executes):",
    violation_result["denied"], "->", violation_result["tool_result"],
)

enterprise_gov = resolve_governance(ENTERPRISE_TENANT, enterprise_session)
last_decision = enterprise_gov.middleware.evidence[-1]
print("\nRecorded decision action:", last_decision.action)
print("Triggered (would-have-been-denied) policies:", last_decision.triggered_policies)
print("Full evidence trail length for this session:", len(enterprise_gov.middleware.evidence))

## 14. Tenant- and session-isolated audit evidence

`TealTigerMiddleware`/`GovernanceBridge` has no native tenant- or session-partitioning of evidence — each instance's `evidence` list is a single flat list, reset in bulk only by `reset_session()`. Isolation here is **structural**, not a notebook-side filter: `resolve_governance` gives every `(tenant_id, session_id)` pair its own `TealTigerMiddleware` instance with its own `GovernanceBridge`, so there is no shared list for anything to leak through in the first place.

In [ ]:
for session_id, gov in _governance_by_session.items():
    tools_seen = sorted({entry["tool_name"] for entry in gov.session_evidence})
    correlation_ids = [entry["correlation_id"] for entry in gov.session_evidence]
    print(
        f"session={session_id!r} tenant={gov.tenant_id!r}: "
        f"{len(gov.session_evidence)} decisions, tools={tools_seen}"
    )
    print(f"  correlation_ids: {correlation_ids}")

## 15. Policy switching without recreating the shared agent

`id(shared_agent)` is captured before and after invoking a fourth, different-tenant call to make the "same agent object" claim executable rather than narrative.

In [ ]:
agent_identity_before = id(shared_agent)

followup = await run_governed_call(
    HEALTHCARE_TENANT, healthcare_session, "lookup_patient_record", {"patient_id": "PT-9001"},
    user_message="Please look up patient PT-9001's latest visit.",
)

agent_identity_after = id(shared_agent)
print("shared_agent identity unchanged:", agent_identity_before == agent_identity_after)
print("Tenants served by this one object so far:", sorted({g.tenant_id for g in _governance_by_session.values()}))

## 16. Assertions

Executable checks against the acceptance criteria, using only values already produced above.

In [ ]:
# One shared agent object served every tenant, never recreated.
assert agent_identity_before == agent_identity_after
assert {g.tenant_id for g in _governance_by_session.values()} == {
    HEALTHCARE_TENANT, FINTECH_TENANT, ENTERPRISE_TENANT,
}

# Tenant selection is driven by tenant_id itself, not a hidden selector.
assert resolve_governance(HEALTHCARE_TENANT, healthcare_session).tenant_id == HEALTHCARE_TENANT

# Healthcare: clean request allowed; PII-containing request blocked; frozen tool denied.
assert clean_call["denied"] is False
assert pii_call["denied"] is True
assert "PII" in pii_call["stopped_reason"].upper()
assert frozen_call["denied"] is True
assert "FREEZE" in frozen_call["stopped_reason"]

# Healthcare audit evidence includes both the PII block and the FREEZE denial.
# process_transaction legitimately appears here: it's the tool the frozen-tool
# call attempted, and its DENY decision is exactly the audit evidence expected.
healthcare_reason_codes = {
    code for entry in healthcare_gov.session_evidence for code in entry["reason_codes"]
}
assert any(code.startswith("PII_") for code in healthcare_reason_codes)
assert "FREEZE_RULE" in healthcare_reason_codes

# Fintech cost cap: first call within budget, second breaches it and is denied.
assert first["denied"] is False
assert second["denied"] is True
assert "cost cap" in second["stopped_reason"]

# Fintech transaction limit: first 3 calls allowed, 4th denied by rate_limit.
assert [r["denied"] for r in tx_results] == [False, False, False, True]
assert "Rate limit" in tx_results[-1]["stopped_reason"]

# Enterprise MONITOR: violation recorded, but the tool call still executed.
assert violation_result["denied"] is False
assert last_decision.action == GovernanceAction.ALLOW
assert "tool_allowlist" in last_decision.triggered_policies

# Tenant/session evidence isolation: no cross-contamination between sessions.
# Each assertion below checks that a tool/stage name belonging to one tenant's
# scenario never appears in a *different* tenant's or session's evidence.
healthcare_tools = {e["tool_name"] for e in _governance_by_session[healthcare_session].session_evidence}
fintech_cost_tools = {e["tool_name"] for e in _governance_by_session[fintech_cost_session].session_evidence}
fintech_tx_tools = {e["tool_name"] for e in _governance_by_session[fintech_tx_session].session_evidence}
enterprise_tools = {e["tool_name"] for e in _governance_by_session[enterprise_session].session_evidence}

assert "<input>" not in fintech_cost_tools  # fintech has no pii policy
assert "<input>" not in fintech_tx_tools  # fintech has no pii policy
assert "<input>" not in enterprise_tools  # enterprise has no pii policy
assert "generate_report" not in healthcare_tools  # healthcare never touches this tool
assert "lookup_patient_record" not in fintech_cost_tools
assert "lookup_patient_record" not in fintech_tx_tools
assert "process_transaction" not in enterprise_tools
assert len(_governance_by_session[fintech_tx_session].session_evidence) == 4
assert healthcare_tools <= {"<input>", "lookup_patient_record", "process_transaction"}
assert enterprise_tools == {"generate_report", "lookup_patient_record"}

print("All acceptance-criteria assertions passed.")

## 17. Production notes and limitations

**Package fixes** (`_engine.py`, `_types.py`; tests in `tests/test_engine.py` / `tests/test_middleware.py`):
- `GovernanceBridge.__init__` accepts `otel_enabled`, so `TealTigerMiddleware(...)` constructs with any documented argument combination.
- `record_tool_success`/`record_tool_failure` implement the `circuit_breaker` policy's failure tracking. Not demonstrated in this notebook, since it isn't part of the tenant scenarios below, but covered by regression tests.
- `GovernanceAction.REDACT` and `GovernanceDecision.redacted_content` were added, and `GovernanceBridge.evaluate_content` is implemented against `tealtiger.guardrails.PIIDetectionGuardrail` — `before_model`/`after_model`/`after_tool` now function as originally written, with no changes to `middleware.py` logic.

**Left unchanged:**
- The `mode` property setter (`set_mode`), `after_agent`'s `finalize_session`, and `export_evidence` still call undefined `GovernanceBridge` methods. This notebook does not call any of them.
- `TealTigerMiddleware.after_tool` is not invoked by LangChain's `AgentMiddleware` framework in `langchain==1.3.16` — there is no `after_tool`/`before_tool` hook in `langchain.agents.middleware.types.AgentMiddleware`, only `before_agent`/`after_agent`/`before_model`/`after_model`/`wrap_model_call`/`wrap_tool_call` and their async variants. This does not affect this notebook, since every hook here is invoked directly rather than through automatic framework dispatch; a `create_agent(...)`-based pipeline would need a different integration point for post-tool-result scanning.
- No HIPAA/fintech/enterprise-branded policy pack exists in `policy-packs/`; the policies above are authored from scratch using supported policy types.
- `rate_limit` is a session-lifetime call-count cap; `window` has no enforcement effect.
- `BudgetScope.type` is `Literal["agent", "project", "organization"]` — there is no `"tenant"` scope, so this notebook reuses `"agent"` with a `tenant::session` composite id.

**Validation:**
- `pytest packages/langchain-tealtiger/tests/` — 65 passed (39 pre-existing, 26 new), 0 failed.
- This notebook runs top-to-bottom against `tealtiger==1.4.0`, `langgraph==1.2.11`, `langchain-core`, and this package installed in editable mode.